Levantando as bases de dados para análise


In [ ]:
import requests
import json
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm
from datetime import datetime
from pathlib import Path


In [ ]:
# Cria função para chamar outras api

# Chama API sem ID   
def extract_api(url):
    try:
        hd = {'accept': 'application/json'}
        response = requests.get(url, headers=hd)
        response.raise_for_status()
        dados = response.json()['dados']
        df = pd.json_normalize(dados)
        return df
    except requests.exceptions.RequestException as e:
        print(f'Erro na requisição{url}: {e}')
        return pd.DataFrame()

#Chama API com ID
def extract_api_append(url, lista_acumuladora):
    """Extrai os dados e adiciona à lista existente."""
    df_dados = extract_api(url) 

    if df_dados is not None and not df_dados.empty:
        lista_acumuladora.append(df_dados)
    
    return lista_acumuladora

# Cria DF usar com API com ID
def make_df(lista_dfs):
    """Concatena a lista em um único DataFrame."""
    if lista_dfs: # Verifica se a lista não está vazia
        return pd.concat(lista_dfs, ignore_index=True)        
    else:
        print('Falha na Extração: Lista vazia.')
        return pd.DataFrame()
    



def save_to_excel(dicionario_dfs, filename="extracao_camara.xlsx", folder='data/processed'):
    if not dicionario_dfs:
        print("Nenhum dado para salvar.")
        return
    output_path = Path(folder)
    output_path.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        for nome_aba, df in dicionario_dfs.items():
            df.to_excel(writer, sheet_name=nome_aba, index=False)
            print(f"Aba '{nome_aba}' gravada com {len(df)} registros.")
            
    print(f"\nArquivo '{filename}' gerado com sucesso.")





In [ ]:
# cria a pasta para armazenar o excel
pasta_data = Path("./data/processed")
pasta_data.mkdir(parents=True, exist_ok=True)
caminho_final = pasta_data / "deputados_detalhado.xlsx"

In [4]:
## Extração Deputados

url = 'https://dadosabertos.camara.leg.br/api/v2/deputados?dataInicio=2016-01-01&ordem=ASC&ordenarPor=nome' # extração desde 2016
df_Deputados = extract_api(url)


In [5]:
df_Deputados.head()

,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,178957,https://dadosabertos.camara.leg.br/api/v2/depu...,ABEL MESQUITA JR.,PMB,https://dadosabertos.camara.leg.br/api/v2/part...,RR,55,https://www.camara.leg.br/internet/deputado/ba...,NaN
1,178957,https://dadosabertos.camara.leg.br/api/v2/depu...,ABEL MESQUITA JR.,DEM,https://dadosabertos.camara.leg.br/api/v2/part...,RR,55,https://www.camara.leg.br/internet/deputado/ba...,NaN
2,220593,https://dadosabertos.camara.leg.br/api/v2/depu...,Abilio Brunini,PL,https://dadosabertos.camara.leg.br/api/v2/part...,MT,57,https://www.camara.leg.br/internet/deputado/ba...,NaN
3,204554,https://dadosabertos.camara.leg.br/api/v2/depu...,Abílio Santana,PHS,https://dadosabertos.camara.leg.br/api/v2/part...,BA,56,https://www.camara.leg.br/internet/deputado/ba...,NaN
4,204554,https://dadosabertos.camara.leg.br/api/v2/depu...,Abílio Santana,PR,https://dadosabertos.camara.leg.br/api/v2/part...,BA,56,https://www.camara.leg.br/internet/deputado/ba...,NaN


In [6]:
repositorio = {'Deputado':df_Deputados}

In [7]:
## Extração Deputados Detalhamento

lista_detalahamentodp =[]

for id in tqdm(df_Deputados['id'], desc='Extract deputados detalhamento'):
   url = f'https://dadosabertos.camara.leg.br/api/v2/deputados/{id}'
   lista_detalahamentodp = extract_api_append(url,lista_detalahamentodp)

df_DetalhamentoDep = make_df(lista_detalahamentodp)
repositorio['DetalhamentoDep'] = df_DetalhamentoDep

Extract deputados detalhamento: 100%|██████████| 1000/1000 [13:45<00:00,  1.21it/s]  


In [12]:
df_DetalhamentoDep.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   id                              1000 non-null   int64 
 1   uri                             1000 non-null   str   
 2   nomeCivil                       1000 non-null   str   
 3   cpf                             1000 non-null   str   
 4   sexo                            1000 non-null   str   
 5   urlWebsite                      8 non-null      object
 6   redeSocial                      1000 non-null   object
 7   dataNascimento                  1000 non-null   str   
 8   dataFalecimento                 6 non-null      object
 9   ufNascimento                    999 non-null    object
 10  municipioNascimento             1000 non-null   str   
 11  escolaridade                    982 non-null    object
 12  ultimoStatus.id                 1000 non-null   int64 
 13  

In [13]:
# Extração de dados legislaturas

url = f'https://dadosabertos.camara.leg.br/api/v2/legislaturas'
df_Legislaturas = extract_api(url)

repositorio['Legislaturas'] = df_Legislaturas

In [14]:
df_Legislaturas.head()

,id,uri,dataInicio,dataFim
0,57,https://dadosabertos.camara.leg.br/api/v2/legi...,2023-02-01,2027-01-31
1,56,https://dadosabertos.camara.leg.br/api/v2/legi...,2019-02-01,2023-01-31
2,55,https://dadosabertos.camara.leg.br/api/v2/legi...,2015-02-01,2019-01-31
3,54,https://dadosabertos.camara.leg.br/api/v2/legi...,2011-02-01,2015-01-31
4,53,https://dadosabertos.camara.leg.br/api/v2/legi...,2007-02-01,2011-01-31


In [15]:
# Extração dados das frentes parlamentares
lista_dfs = []

for id in tqdm(df_Deputados['id'], desc='Extract Deputados'):
    # Coleta os dados do deputado atual
    url = f'https://dadosabertos.camara.leg.br/api/v2/deputados/{id}/frentes'
    lista_dfs = extract_api_append(url,lista_dfs)
    
df_FrentesParlamentares = make_df(lista_dfs)

repositorio['FrentesParlamentares'] = df_FrentesParlamentares



Extract Deputados: 100%|██████████| 1000/1000 [16:20<00:00,  1.02it/s] 


In [16]:
df_FrentesParlamentares.head()

,id,uri,titulo,idLegislatura
0,53857,https://dadosabertos.camara.leg.br/api/v2/fren...,Requer o registro da Frente Parlamentar em Def...,55
1,53842,https://dadosabertos.camara.leg.br/api/v2/fren...,Frente Parlamentar em Defesa do Saneamento Bás...,55
2,53834,https://dadosabertos.camara.leg.br/api/v2/fren...,Frente Parlamentar Mista Nacional da Indústria...,55
3,53831,https://dadosabertos.camara.leg.br/api/v2/fren...,Frente Parlamentar de Proteção de Dados Pessoais,55
4,53829,https://dadosabertos.camara.leg.br/api/v2/fren...,Frente Parlamentar Mista em Apoio aos Planos d...,55


In [17]:
url = f'https://dadosabertos.camara.leg.br/api/v2/partidos'
df_Partidos = extract_api(url)

repositorio['df_Partidos'] = df_Partidos

In [18]:
df_Partidos.head(5)

,id,sigla,nome,uri
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/part...
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/part...
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/part...
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/part...
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/part...


In [19]:
lista_dfs = []

for id in tqdm(df_Partidos['id'], desc= 'extrect partidos detalhamento'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/partidos/{id}'
    lista_dfs = extract_api_append(url, lista_dfs)

df_PartidosDetalhado = make_df(lista_dfs)

repositorio['PartidosDetalhado'] = df_PartidosDetalhado

extrect partidos detalhamento: 100%|██████████| 15/15 [00:04<00:00,  3.28it/s]


In [20]:
df_PartidosDetalhado.head(5)

,id,sigla,nome,uri,numeroEleitoral,urlLogo,urlWebSite,urlFacebook,status.data,status.idLegislatura,...,status.totalPosse,status.totalMembros,status.uriMembros,status.lider.uri,status.lider.nome,status.lider.siglaPartido,status.lider.uriPartido,status.lider.uf,status.lider.idLegislatura,status.lider.urlFoto
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2026-02-19T15:27,57,...,7,8,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Bruno Farias,AVANTE,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2021-03-02T13:23,57,...,5,4,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Alex Manente,CIDADANIA,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2024-10-07T08:39,57,...,41,43,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Isnaldo Bulhões Jr.,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AL,57,https://www.camara.leg.br/internet/deputado/ba...
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2026-03-10T12:14,57,...,0,1,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Kim Kataguiri,MISSÃO,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2025-04-24T19:20,57,...,3,5,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Marcel van Hattem,NOVO,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...


In [21]:
# Extração de proposições
hoje =datetime.now()
lista_dfs = []
for ano in tqdm(range(2022,2027), desc='Extract proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano={ano}&ordem=ASC&ordenarPor=id'
    lista_dfs = extract_api_append(url, lista_dfs)

df_Proposicoes = make_df(lista_dfs)

repositorio['Proposicoes'] = df_Proposicoes

Extract proposições:   0%|          | 0/5 [00:00<?, ?it/s]

Extract proposições: 100%|██████████| 5/5 [00:15<00:00,  3.01s/it]


In [22]:
df_Proposicoes.head()

,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,303153,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,618,2022,Dispõe sobre o exercício da profissão de Podól...,2005-10-11T14:41
1,493361,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,2253,2022,Dispõe sobre o monitoramento por instrumentos ...,2011-02-23T19:42
2,587207,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1367,2022,Dispõe sobre a prestação dos serviços de contr...,2013-08-14T18:06
3,597587,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,3062,2022,"Altera a redação dos arts. 14, 17 e 18 da Lei ...",2013-10-22T10:53
4,996806,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1637,2022,Dispõe sobre a avaliação psicológica de gestan...,2015-03-12T15:02


In [23]:
df_Proposicoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                75 non-null     int64
 1   uri               75 non-null     str  
 2   siglaTipo         75 non-null     str  
 3   codTipo           75 non-null     int64
 4   numero            75 non-null     int64
 5   ano               75 non-null     int64
 6   ementa            75 non-null     str  
 7   dataApresentacao  75 non-null     str  
dtypes: int64(4), str(4)
memory usage: 4.8 KB


In [27]:
#Extração detalhamento da proposição
lista_dfs = []

for i in tqdm(df_Proposicoes ['id'], desc= 'Extract Detalhamento Proposição'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{i}'
    lista_dfs = extract_api_append(url, lista_dfs)

df_ProposicoesDetalhamento = make_df(lista_dfs)

repositorio['DetalhamentoProp'] = df_ProposicoesDetalhamento




Extract Detalhamento Proposição: 100%|██████████| 75/75 [00:16<00:00,  4.50it/s]


In [28]:
df_ProposicoesDetalhamento.head()

,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao,uriOrgaoNumerador,uriAutores,...,statusProposicao.uriUltimoRelator,statusProposicao.regime,statusProposicao.descricaoTramitacao,statusProposicao.codTipoTramitacao,statusProposicao.descricaoSituacao,statusProposicao.codSituacao,statusProposicao.despacho,statusProposicao.url,statusProposicao.ambito,statusProposicao.apreciacao
0,303153,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,618,2022,Dispõe sobre o exercício da profissão de Podól...,2005-10-11T14:41,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/prop...,...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Apresentação de Requerimento,194,Pronta para Pauta,924,Apresentação do REQ n. 3824/2025 (Requerimento...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
1,493361,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,2253,2022,Dispõe sobre o monitoramento por instrumentos ...,2011-02-23T19:42,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/prop...,...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Urgência (Art. 155, RICD)",Apresentação de Proposição,100,Transformado em Norma Jurídica,1140,Recebido Ofício 219/2024-CN que comunica resti...,None,Regimental,Proposição Sujeita à Apreciação do Plenário
2,587207,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1367,2022,Dispõe sobre a prestação dos serviços de contr...,2013-08-14T18:06,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/prop...,...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Parecer do(a) Relator(a),322,Pronta para Pauta,924,Parecer à(s) Emenda(s) / ao Substitutivo do Se...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
3,597587,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,3062,2022,"Altera a redação dos arts. 14, 17 e 18 da Lei ...",2013-10-22T10:53,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/prop...,...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Urgência (Art. 155, RICD)",Transformação em Norma Jurídica,251,Transformado em Norma Jurídica,1140,Transformado na Lei Ordinária 15183/2025. DOU ...,None,Regimental,Proposição Sujeita à Apreciação do Plenário
4,996806,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1637,2022,Dispõe sobre a avaliação psicológica de gestan...,2015-03-12T15:02,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/prop...,...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Designação de Relator(a),320,Aguardando Parecer,915,"Designada Relatora, Dep. Sâmia Bomfim (PSOL-SP).",None,Regimental,Proposição Sujeita à Apreciação do Plenário


In [29]:
df_ProposicoesDetalhamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   id                                    75 non-null     int64 
 1   uri                                   75 non-null     str   
 2   siglaTipo                             75 non-null     str   
 3   codTipo                               75 non-null     int64 
 4   numero                                75 non-null     int64 
 5   ano                                   75 non-null     int64 
 6   ementa                                75 non-null     str   
 7   dataApresentacao                      75 non-null     str   
 8   uriOrgaoNumerador                     38 non-null     object
 9   uriAutores                            75 non-null     str   
 10  descricaoTipo                         75 non-null     str   
 11  ementaDetalhada                       54 non-

In [30]:
#Extração autores das proposições

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract autores das proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/autores'
    lista_dfs = extract_api_append(url, lista_dfs)

df_AutoresProposicoes = make_df(lista_dfs)

repositorio['AutoresProposicoes'] = df_AutoresProposicoes

Extract autores das proposições: 100%|██████████| 75/75 [02:42<00:00,  2.17s/it]


In [31]:
df_AutoresProposicoes.head()

,uri,nome,codTipo,tipo,ordemAssinatura,proponente
0,https://dadosabertos.camara.leg.br/api/v2/depu...,José Mentor,10000,Deputado(a),1,1
1,https://dadosabertos.camara.leg.br/api/v2/depu...,Pedro Paulo,10000,Deputado(a),1,1
2,https://dadosabertos.camara.leg.br/api/v2/depu...,Laercio Oliveira,10000,Deputado(a),1,1
3,https://dadosabertos.camara.leg.br/api/v2/depu...,Ricardo Izar,10000,Deputado(a),1,1
4,https://dadosabertos.camara.leg.br/api/v2/depu...,Célio Silveira,10000,Deputado(a),1,1


In [34]:
ordemAssinatura = df_AutoresProposicoes['ordemAssinatura'].unique()
proponente = df_AutoresProposicoes['proponente'].unique()
print(f'Possivéis ordem de assinatura {ordemAssinatura}\nPossíveis prponente: {proponente}')

Possivéis ordem de assinatura [1 2]
Possíveis prponente: [1]


In [33]:
df_AutoresProposicoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   uri              73 non-null     str  
 1   nome             73 non-null     str  
 2   codTipo          73 non-null     int64
 3   tipo             73 non-null     str  
 4   ordemAssinatura  73 non-null     int64
 5   proponente       73 non-null     int64
dtypes: int64(3), str(3)
memory usage: 3.6 KB


In [35]:
# Extração tipos de proposição

url = 'https://dadosabertos.camara.leg.br/api/v2/referencias/tiposProposicao'
df_TiposProposicao = extract_api(url)

repositorio['df_TiposProposicao'] = df_TiposProposicao

In [36]:
df_TiposProposicao.head()

,cod,sigla,nome,descricao
0,27,OF,Ofício do Congresso Nacional,
1,129,CON,Consulta,Consulta
2,130,EMC,Emenda na Comissão,Emenda Apresentada na Comissão
3,131,EMP,Emenda de Plenário,Emenda de Plenário
4,132,EMS,Emenda/Substitutivo do Senado,Emenda/Substitutivo do Senado


In [37]:
# estração de temas das proposições 

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract temas das proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/temas'
    lista_dfs = extract_api_append(url, lista_dfs)
df_TemasProposicoes = make_df(lista_dfs)

repositorio['TemasProposicoes'] = df_TemasProposicoes

Extract temas das proposições: 100%|██████████| 75/75 [02:40<00:00,  2.14s/it]


In [38]:
df_TemasProposicoes.head()

,codTema,tema,relevancia
0,58,Trabalho e Emprego,0
1,43,Direito Penal e Processual Penal,0
2,66,"Indústria, Comércio e Serviços",0
3,56,Saúde,0
4,48,Meio Ambiente e Desenvolvimento Sustentável,0


In [39]:
# estração de trramitação das proposições 

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract tramitação das proposicoes'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/tramitacoes'
    lista_dfs = extract_api_append (url, lista_dfs)

df_TramitacaoProposicao = make_df(lista_dfs)

repositorio['TramitacaoProposicao'] = df_TramitacaoProposicao

Extract tramitação das proposicoes: 100%|██████████| 75/75 [00:18<00:00,  4.01it/s]


In [40]:
df_TramitacaoProposicao.head()

,dataHora,sequencia,siglaOrgao,uriOrgao,uriUltimoRelator,regime,descricaoTramitacao,codTipoTramitacao,descricaoSituacao,codSituacao,despacho,url,ambito,apreciacao
0,2005-10-11T14:41,1,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Apresentação de Proposição,100,Pronta para Pauta,924,Apresentação do Projeto de Lei pelo Deputado J...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
1,2005-10-20T15:28,10,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Distribuição,110,Aguardando análise de prazo recursal,1050,Às Comissões de Seguridade Social e Família; T...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
2,2005-10-25T00:00,15,CCP,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Publicação de Proposição,604,Aguardando Encaminhamento,910,Encaminhada à publicação. Publicação Inicial n...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
3,2005-12-01T00:00,17,CSAUDE,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Designação de Relator(a),320,Aguardando Encaminhamento,910,"Designada Relatora, Dep. Almerinda de Carvalho...",NaN,Regimental,Proposição Sujeita à Apreciação do Plenário
4,2005-12-02T00:00,18,CSAUDE,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Abertura de Prazo,350,Aguardando Encaminhamento,910,Prazo para Emendas ao Projeto (5 sessões ordin...,NaN,Regimental,Proposição Sujeita à Apreciação do Plenário


In [41]:
#Extração tipos de tramitação

url = 'https://dadosabertos.camara.leg.br/api/v2/referencias/tiposTramitacao'
df_TiposTramitacao = extract_api(url)

repositorio['df_TiposTramitacao'] = df_TiposTramitacao

In [42]:
# extracao de votações

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract votações propositação'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/votacoes?ordem=DESC&ordenarPor=dataHoraRegistro'
    lista_dfs = extract_api_append(url,lista_dfs)

df_VotacoesProposicoes = make_df(lista_dfs)

repositorio['VotacoesProposicoes'] = df_VotacoesProposicoes

Extract votações propositação:   0%|          | 0/75 [00:00<?, ?it/s]

Extract votações propositação: 100%|██████████| 75/75 [03:44<00:00,  2.99s/it]


In [43]:
lista_dfs =[]

for a in tqdm(range(2022,2027), desc='Extract votações'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/votacoes?dataInicio={a}-01-01&ordem=DESC&ordenarPor=dataHoraRegistro'
    lista_dfs = extract_api_append(url,lista_dfs)

df_VotacoesAnos= make_df(lista_dfs)

repositorio['VotacoesAnos'] = df_VotacoesAnos

Extract votações: 100%|██████████| 5/5 [00:16<00:00,  3.39s/it]


In [44]:
df_VotacoesAnos.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   500 non-null    str    
 1   uri                  500 non-null    str    
 2   data                 500 non-null    str    
 3   dataHoraRegistro     500 non-null    str    
 4   siglaOrgao           500 non-null    str    
 5   uriOrgao             500 non-null    str    
 6   uriEvento            383 non-null    str    
 7   proposicaoObjeto     221 non-null    str    
 8   uriProposicaoObjeto  221 non-null    str    
 9   descricao            500 non-null    str    
 10  aprovacao            493 non-null    float64
dtypes: float64(1), str(10)
memory usage: 43.1 KB


In [45]:
# esxtração cod Tipo de Tramitação

url=  'https://dadosabertos.camara.leg.br/api/v2/referencias/proposicoes/codTipoTramitacao'
df_CodTiposTramitacao = extract_api(url)

repositorio['CodTiposTramitacao'] = df_CodTiposTramitacao

In [46]:
# Detalhamento Das votações

 
lista_dfs = []

for v in tqdm(df_VotacoesAnos['id'], desc= 'Extract votações detalhamento'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/votacoes/{v}'
    lista_dfs = extract_api_append(url, lista_dfs)

df_VotacoesDetalhamento = make_df(lista_dfs)



Extract votações detalhamento:  83%|████████▎ | 416/500 [09:19<00:19,  4.30it/s]  

Erro na requisiçãohttps://dadosabertos.camara.leg.br/api/v2/votacoes/2291805-49: 404 Client Error: Not Found for url: https://dadosabertos.camara.leg.br/api/v2/votacoes/2291805-49


Extract votações detalhamento:  84%|████████▍ | 422/500 [09:21<00:19,  4.03it/s]

Erro na requisiçãohttps://dadosabertos.camara.leg.br/api/v2/votacoes/2291805-44: 404 Client Error: Not Found for url: https://dadosabertos.camara.leg.br/api/v2/votacoes/2291805-44


Extract votações detalhamento: 100%|██████████| 500/500 [09:37<00:00,  1.16s/it]


In [47]:

df_VotacoesDetalhamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 19 columns):
 #   Column                                            Non-Null Count  Dtype 
---  ------                                            --------------  ----- 
 0   id                                                498 non-null    str   
 1   uri                                               498 non-null    str   
 2   data                                              498 non-null    str   
 3   dataHoraRegistro                                  498 non-null    str   
 4   siglaOrgao                                        498 non-null    str   
 5   uriOrgao                                          498 non-null    str   
 6   idOrgao                                           498 non-null    int64 
 7   uriEvento                                         383 non-null    object
 8   idEvento                                          383 non-null    object
 9   descricao                                  

In [48]:
df_VotacoesDetalhamento.head()

,id,uri,data,dataHoraRegistro,siglaOrgao,uriOrgao,idOrgao,uriEvento,idEvento,descricao,aprovacao,descUltimaAberturaVotacao,dataHoraUltimaAberturaVotacao,efeitosRegistrados,objetosPossiveis,proposicoesAfetadas,ultimaApresentacaoProposicao.dataHoraRegistro,ultimaApresentacaoProposicao.descricao,ultimaApresentacaoProposicao.uriProposicaoCitada
0,2314271-15,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09,2022-04-05T14:48:18,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,180,https://dadosabertos.camara.leg.br/api/v2/even...,64653,Aprovada a Redação Final assinada pelo Relator...,1,Votação da Redação Final.,2022-04-05T14:48:05,[],"[{'id': 2314871, 'uri': 'https://dadosabertos....","[{'id': 2314271, 'uri': 'https://dadosabertos....",2022-02-09T15:50:34,Apresentação do Projeto de Decreto Legislativo...,None
1,2314271-13,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-02-09,2022-04-05T14:47:50,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,180,https://dadosabertos.camara.leg.br/api/v2/even...,64653,Aprovado o Projeto de Decreto Legislativo nº 2...,1,None,None,[],"[{'id': 2314871, 'uri': 'https://dadosabertos....","[{'id': 2314271, 'uri': 'https://dadosabertos....",2022-02-09T15:50:34,Apresentação do Projeto de Decreto Legislativo...,None
2,2318954-16,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-03-30,2022-04-01T10:59:03,GTPL3890,https://dadosabertos.camara.leg.br/api/v2/orga...,539004,https://dadosabertos.camara.leg.br/api/v2/even...,64797,"Aprovado o Relatório do Relator, Dep. Gilberto...",1,Iniciada a votação da matéria.,2022-03-30T15:36:01,[],"[{'id': 2319039, 'uri': 'https://dadosabertos....","[{'id': 2318954, 'uri': 'https://dadosabertos....",None,"Apresentação da Relatório n. 3/2022, pelo Depu...",None
3,1701050-119,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-03-31,2022-03-31T13:26:30,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,4,None,None,Deferido.,1,None,None,[],"[{'id': 2318830, 'uri': 'https://dadosabertos....","[{'id': 1701050, 'uri': 'https://dadosabertos....",None,Apresentação do Requerimento de Retirada de Pr...,https://dadosabertos.camara.leg.br/api/v2/prop...
4,2317281-19,https://dadosabertos.camara.leg.br/api/v2/vota...,2022-03-31,2022-03-31T11:06:02,SECAP(SGM),https://dadosabertos.camara.leg.br/api/v2/orga...,100001,None,None,Realizar o encaminhamento do PL-454/2022 à CE ...,1,None,None,[],[],"[{'id': 2317281, 'uri': 'https://dadosabertos....",None,None,None


In [ ]:
# cria um excel com todos os DF

save_to_excel(repositorio)


Aba 'Deputado' gravada com 1000 registros.
Aba 'DetalhamentoDep' gravada com 1000 registros.
Aba 'Proposicoes' gravada com 75 registros.
Aba 'Legislaturas' gravada com 15 registros.


/mnt/d/GabrielaTorres/estudos/univesp/Univesp_Projetos/dashboard_camara/pi_camara_deputados/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://dadosabertos.camara.leg.br/api/v2/frentes/54309' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/mnt/d/GabrielaTorres/estudos/univesp/Univesp_Projetos/dashboard_camara/pi_camara_deputados/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://dadosabertos.camara.leg.br/api/v2/frentes/54325' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/mnt/d/GabrielaTorres/estudos/univesp/Univesp_Projetos/dashboard_camara/pi_camara_deputados/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://dadosabertos.camara.leg.br/api/v2/frentes/54327' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/mnt/d/GabrielaTorres/estudos/univesp/Univesp_Projetos/dashboard_camar

Aba 'FrentesParlamentares' gravada com 272775 registros.
Aba 'df_Partidos' gravada com 15 registros.
Aba 'PartidosDetalhado' gravada com 15 registros.
Aba 'DetalhamentoProp' gravada com 75 registros.
Aba 'AutoresProposicoes' gravada com 73 registros.
Aba 'df_TiposProposicao' gravada com 544 registros.
Aba 'TemasProposicoes' gravada com 86 registros.
Aba 'TramitacaoProposicao' gravada com 2521 registros.
Aba 'df_TiposTramitacao' gravada com 233 registros.
Aba 'VotacoesProposicoes' gravada com 338 registros.
Aba 'VotacoesAnos' gravada com 500 registros.
Aba 'CodTiposTramitacao' gravada com 233 registros.

Arquivo 'extracao_camara.xlsx' gerado com sucesso.
